# Chit 5 baseline: a mosquito detector for a bed-net sensor · **MCU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

A battery-powered sensor by a bed net listens all night and counts mosquitoes, so that only tiny nightly summaries leave the device. Train a small CNN on the paper's log-mel features of 1.92-s audio windows, measure **mosquito recall at ≤ 1% false alarms** and PR-AUC on **unseen sites** (Tanzania field, UK lab cultures) in fp32 and int8, and count the cost **including the feature extraction**.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime", "soundfile"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 1. Data
One sample = 1.92 s of audio at 8 kHz (15,360 int16 samples); label 1 = mosquito. `test` holds two whole experiments the model never sees in training:
**A** = field recordings in Tanzania, **B** = lab cultures in the UK (`meta("test")`). No recording is in two splits.

In [ ]:
from capstone.data.humbug import load, meta, logmel, logmel_macs, recall_at_false_alarm, pr_auc, WINDOW, SR
A_tr, y_tr = load("train", max_items=4_000 if FAST else None)
A_va, y_va = load("val"); A_te, y_te = load("test")
TEST_SET = meta("test")["test_set"]
for name, y in (("train", y_tr), ("val", y_va), ("test", y_te), ("test A", y_te[TEST_SET == "A"]), ("test B", y_te[TEST_SET == "B"])):
    print(f"{name:<6} {len(y):>7,} windows ({len(y) * WINDOW / SR / 3600:5.1f} h), {np.mean(y):.1%} mosquito")

## 2. Features: the paper's log-mel spectrogram, and what it costs
`logmel` turns each window into 128 mel bands × 30 frames (2048-point FFT, hop 512). On the device this runs for every window, so its
multiply-accumulates count toward the budget: `logmel_macs()`. FFT size, hop, number of bands and sample rate are all levers.

In [ ]:
t0 = time.time()
XTR, XVA, XTE = logmel(A_tr), logmel(A_va), logmel(A_te)
print(f"features {XTR.shape[1:]} for {len(XTR) + len(XVA) + len(XTE):,} windows in {time.time() - t0:.0f} s; {logmel_macs():,} MACs per window")
A_one = A_te[:1].copy()
del A_tr, A_va, A_te                              # the raw audio is no longer needed: free the memory

## 3. The model, costed before training
Three strided 3 × 3 convolutions (128 × 30 → 16 × 4), global average pooling, a linear layer. `extra_macs=logmel_macs()` adds the features.

In [ ]:
model = nn.Sequential(nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
                      nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2))
p = profile(model, (1, 128, 30), extra_macs=logmel_macs())
print(f"params {p['params']:,} | MACs {p['macs']:,} (of which log-mel {logmel_macs():,}) | int8 weights {p['weight_bytes']:,} B | peak act {p['peak_act_bytes']:,} B")
WIN_S = WINDOW / SR
print(f"latency at 100 M MAC/s: {1e3 * mcu_latency(p['macs'], 100e6):.1f} ms per {WIN_S:.2f}-s window "
      f"(the device is busy {mcu_latency(p['macs'], 100e6) / WIN_S:.1%} of the time)")
E = energy_uj(p)
print(f"energy (card formula): {E:.2f} µJ per window = {E * 3600 / WIN_S / 1e3:.2f} mJ per hour of listening")
TARGET = dict(max_weight_bytes=16 * 1024, max_act_bytes=32 * 1024, mac_rate=50e6, max_latency_s=0.100)   # PROVISIONAL: your chit card has the budgets
fits(p, TARGET);

## 4. Train and measure
The threshold is set on the test set's no-mosquito windows so that 1% of them raise a false alarm; recall is measured at that threshold.

In [ ]:
fit(model, XTR, y_tr, epochs=1 if FAST else 10, bs=128, lr=3e-3)
def scores(out):
    return torch.softmax(torch.as_tensor(out), 1)[:, 1].numpy()
s32 = scores(predict(model, XTE))
REC32, AP32 = recall_at_false_alarm(s32, y_te), pr_auc(s32, y_te)
sva = scores(predict(model, XVA))
print(f"val (sites seen in training): recall @ 1% false alarms {recall_at_false_alarm(sva, y_va):.3f}, PR-AUC {pr_auc(sva, y_va):.3f}")
print(f"test (unseen sites), fp32:    recall @ 1% false alarms {REC32:.3f}, PR-AUC {AP32:.3f}")

## 5. int8 (weights and activations), with ONNX Runtime static quantization
An MCU runs int8: this is the number the chit counts.

In [ ]:
to_onnx(model.cpu(), XTR[:1], "my_models/humbug_fp32.onnx")
ort_int8("my_models/humbug_fp32.onnx", XTR[np.random.default_rng(0).permutation(len(XTR))[:512]], "my_models/humbug_int8.onnx")
sess8 = ort_session("my_models/humbug_int8.onnx")
OUT8 = ort_predict(sess8, XTE)

s8 = scores(OUT8)
REC8, AP8 = recall_at_false_alarm(s8, y_te), pr_auc(s8, y_te)
print(f"test, int8: recall @ 1% false alarms {REC8:.3f}, PR-AUC {AP8:.3f}   <- the chit's metric")
for ts in ("A", "B"):
    m = TEST_SET == ts
    print(f"  test {ts}: recall @ 1% false alarms {recall_at_false_alarm(s8[m], y_te[m]):.3f}, PR-AUC {pr_auc(s8[m], y_te[m]):.3f}")
print("other false-alarm rates (int8): " + ", ".join(f"{fa:.1%} -> recall {recall_at_false_alarm(s8, y_te, fa):.3f}" for fa in (0.05, 0.02, 0.005)))

**What 1% false alarms means in the field.** The test set is about half mosquito; a real night is ~1-2% mosquito. With prevalence π, recall r and
false-alarm rate f, the fraction of the device's detections that are real mosquitoes is r·π / (r·π + f·(1 − π)):

In [ ]:
for prev in (0.01, 0.02):
    print(f"at {prev:.0%} mosquito time: {REC8 * prev / (REC8 * prev + 0.01 * (1 - prev)):.0%} of the detections are real")

## 6. Sanity measurement (required for MCU chits): ONNX Runtime int8 on 1 thread
Far faster than a microcontroller; what matters is whether your candidates **rank** like their MAC counts. The log-mel is timed too (NumPy here):
it is half the MACs, but it is FFTs and a log, not a convolution.

In [ ]:
from common.timing import measure
LAT8 = ort_latency(sess8, XTE[:1])
LATF = measure(lambda: logmel(A_one))
print(f"CNN int8, ORT 1 thread: median {LAT8['median_us']:.0f} µs, p99 {LAT8['p99_us']:.0f} µs | log-mel (NumPy): median {LATF['median_us']:.0f} µs, p99 {LATF['p99_us']:.0f} µs")

In [ ]:
ROW = report("5 humbug", (len(XTR), len(XVA), len(XTE)), (1, 128, 30),
             f"recall@1%FA int8 {REC8:.3f} PR-AUC {AP8:.3f} (fp32 {REC32:.3f}/{AP32:.3f})", p,
             f"MCU counted: {1e3 * mcu_latency(p['macs'], 100e6):.1f} ms per 1.92-s window at 100 M MAC/s incl. log-mel; "
             f"ORT int8 1 thread p99 {LAT8['p99_us']:.0f} µs (CNN only); energy {E:.2f} µJ/window",
             "CC BY 4.0 (Zenodo, confirmed)", extra=dict(logmel_macs=logmel_macs(), energy_uj=round(E, 3), minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.